In [1]:
import openai
from openai import OpenAI
import tiktoken
import logging
import pandas as pd

In [2]:
DEFAULT_SYSTEM_MESSAGE = """You are a scientific visualization expert and prompt engineer.

Create a prompt for an AI image generator that will produce an accurate and visually compelling cover image for a scientific paper.

CRITICAL RULES:
- Scientific accuracy: Visual elements must correctly represent the scientific concept
- No fictional or decorative elements that contradict the research
- Use correct scientific terminology in the prompt
- Output ONLY the prompt, no extra text

PROMPT COMPONENTS:
1. Core scientific concept (what is being shown)
2. Specific visual details (colors, materials, lighting)
3. Style appropriate for the journal/conference
4. Composition and perspective

EXAMPLES:

Article: "Graphene-Based Battery with 10x Capacity"
Prompt: "Cross-section of graphene layered anode material, lithium ions intercalating between graphene sheets, electron flow visualized as golden energy streams, blue and gray color palette, scientific illustration style, cutaway view, detailed material texture"

Article: "Neural Network Explains Visual Cortex Activity"
Prompt: "Artificial neural network architecture overlaid on primate visual cortex diagram, activation patterns shown as colored heatmaps, connections between nodes mimicking biological pathways, academic illustration style, split-view showing both AI and biology, cool blue to warm red gradient
"""

In [ ]:
class Teacher:
    def __init__(self, api_key, model='deepseek-chat', temperature=0.7, max_tokens=150, n=3, system_message=DEFAULT_SYSTEM_MESSAGE, debug=False):
        self.client = OpenAI(api_key=api_key, base_url='https://api.deepseek.com')
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.n = n

        self.in_tokens = 0
        self.out_tokens = 0
        self.total_cost = 0

        self.tiktoken_encoder = tiktoken.get_encoding('cl100k_base')
        
        self.system_message = {
            "role" : "system", 
            "content" : system_message
        }

        logging.basicConfig(
            level=logging.INFO if not debug else logging.DEBUG,
            format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
            handlers=[logging.FileHandler('../logs/teacher.log'), logging.StreamHandler()]
        )
        self.logger = logging.getLogger(__name__)

        self.logger.debug(f"Teacher initialized with model={model}, temp={temperature}, n={n}")

    def _call_api(self, messages):
        try:
            response = self.client.chat.completions.create(
                model = self.model,
                messages=messages,
                temperature=self.temperature,
                max_tokens=self.max_tokens,
                n=self.n
            )
            self.logger.info(f'Response received, choices={len(response.choices)}')
            return response
        except Exception as e:
            self.logger.error(f"An error accured during calling API: {e}")
            raise e

    def _prepare_messages(self, title, abstract):
       user_content = f"Generate a prompt for:\nTitle: {title}\n\nAbstract: {abstract}" 
       messages = [self.system_message, {"role" : "user", "content" : user_content}]
       self.logger.debug(f"Prepared messages. System length: {len(self.system_message['content'])}, user length: {len(user_content)}")
       return messages
    
    def _update_stats(self, response):
        try:
            in_t = response.usage.prompt_tokens
            out_t = response.usage.completion_tokens
        except Exception as e:
            self.logger.error(f"An error occured during stats updating: {e}")
            raise

        self.in_tokens += in_t
        self.out_tokens += out_t

        cost = (in_t * 0.14 + out_t * 0.28) / 1_000_000
        self.total_cost += cost

        self.logger.info(f"Stats. Tokens: {in_t} + {out_t} = {in_t + out_t}, cost: {cost}, total cost: {self.total_cost}.")
        self.logger.debug(f"Global stats. Tokens: {self.in_tokens}, {self.out_tokens}, cost: {self.total_cost}")

    def generate(self, article):
        self.logger.info(f"Start generating prompts for {article['id']}")
        self.logger.debug(f"Article: id={article['id']}, title_len={article['title']}, abstract_len={article['abstract']}")

        try:
            messages = self._prepare_messages(article['title'], article['abstract'])
            response = self._call_api(messages)
            self._update_stats(response)

            output = pd.DataFrame({'id' : [article['id']]})
            for i, choice in enumerate(response.choices):
                output[f'prompt #{i}'] = choice.message.content
            
            return output
        except Exception as e:
            self.logger.error(f'An error occured during response: {e}')
            raise

    def generate_from_dataset(self, dataset, save=True, save_path="promts.tsv", save_mode='w', checkpoint_path=None, checkpoint_every=100):
        all_res = []
        total = dataset.shape[0]
        self.logger.info(f"Starting processing {total} articles.")
        for i, article in dataset.iterrows():
            self.logger.info(f"Processing {i+1}/{total}: {article['id']}")
            try:
               res = self.generate(article)
               all_res.append(res)
               if checkpoint_path and (i + 1) % checkpoint_every == 0:
                   self._save_checkpoint(all_res, checkpoint_path)
            except Exception as e:
                self.logger.error(f"Failed to process {article['id']}: {e}")
                continue   
        if checkpoint_path:
            self._save_checkpoint(all_res, checkpoint_path)
            self.logger.info(f"Saved final results to {checkpoint_path}")
        df_fin = pd.concat(all_res, ignore_index=True)
        self.logger.info(f"Completed. Processed {df_fin.shape[0]} articles")
        if save:
            self.logger.info("Saving dataframe.")
            df_fin.to_csv(save_path, sep='\t', mode=save_mode, header=(save_mode == 'w'), index=False, encoding='utf-8-sig')
            self.logger.info("Succeed.")
        return df_fin
                

In [ ]:
teacher = Teacher(
    api_key='',
    model='deepseek-chat',
    temperature=0.7,
    max_tokens=150,
    system_message=DEFAULT_SYSTEM_MESSAGE,
    debug=True
)
data = pd.read_csv("../dataset/clean.tsv", sep='\t')
teacher.generate_from_dataset(
    dataset=data,
    save=True,
    save_path="../dataset/promts.tsv",
    save_mode="w",
    checkpoint_path="../checkpoint",
    checkpoint_every=100
)

254